# Análisis Exploratorio — CommonLit: Evaluar resúmenes de estudiantes

Predicción de las puntuaciones `content` y `wording` de resúmenes escritos por estudiantes.
Competencia: https://www.kaggle.com/competitions/commonlit-evaluate-student-summaries/overview



El objetivo es predecir dos puntuaciones que un evaluador experto asigna al resumen de un estudiante:

- **content**: qué tan bien el resumen captura y relaciona las ideas principales del texto fuente.
- **wording**: claridad, precisión y calidad objetiva de la redacción (vocabulario y sintaxis propios).

La métrica de la competencia es MCRMSE.

Solo hay 4 textos fuente (*prompts*) en entrenamiento y el test real usa prompts distintos, así que el problema central es generalizar a prompts no vistos. Por eso la validación debe hacerse por grupo (`GroupKFold` sobre `prompt_id`).


In [ ]:
import sys, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / 'src'))
import eda_utils as eu

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')

DATA_DIR = Path.cwd().parent / 'data'
FIG_DIR = Path.cwd().parent / 'figuras'
FIG_DIR.mkdir(exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')

print('Archivos en data/:', sorted(p.name for p in DATA_DIR.glob('*')))

## Carga de datos

In [ ]:
summaries_train = pd.read_csv(DATA_DIR / 'summaries_train.csv')
prompts_train = pd.read_csv(DATA_DIR / 'prompts_train.csv')
summaries_test = pd.read_csv(DATA_DIR / 'summaries_test.csv')
prompts_test = pd.read_csv(DATA_DIR / 'prompts_test.csv')

print('summaries_train:', summaries_train.shape)
print('prompts_train  :', prompts_train.shape)
print('summaries_test :', summaries_test.shape)
print('prompts_test   :', prompts_test.shape)
summaries_train.head()

## Variables y observaciones

Número de filas y columnas de cada tabla, tipo de cada variable, faltantes y cardinalidad.

In [ ]:
def describe_schema(df, name):
    info = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'n_missing': df.isna().sum(),
        'pct_missing': (df.isna().mean() * 100).round(2),
        'n_unique': df.nunique(),
        'ejemplo': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })
    print(f'== {name} ==  filas={len(df)}  columnas={df.shape[1]}')
    return info

describe_schema(summaries_train, 'summaries_train')

In [ ]:
describe_schema(prompts_train, 'prompts_train')

In [ ]:
# Textos fuente: consigna y longitud de cada prompt
prompts_train.assign(
    palabras_texto_fuente=prompts_train['prompt_text'].map(lambda t: len(eu.tokenize_words(t))),
    palabras_consigna=prompts_train['prompt_question'].map(lambda t: len(eu.tokenize_words(str(t)))),
)[['prompt_id', 'prompt_title', 'prompt_question', 'palabras_texto_fuente', 'palabras_consigna']]

In [ ]:
# Duplicados y consistencia de llaves
print('student_id duplicados:', summaries_train['student_id'].duplicated().sum())
print('prompt_id en summaries no presentes en prompts:',
      set(summaries_train['prompt_id']) - set(prompts_train['prompt_id']))
print('resúmenes por prompt:')
print(summaries_train['prompt_id'].value_counts())

**Observaciones de la descripción de los datos**

- `summaries_train` tiene 7 165 filas y 5 columnas; prompts_train tiene 4 filas y 4 columnas. El test.csv visible es solo un formato.
- Tipos: `student_id` y `prompt_id` son texto (identificador y categórica); `text` es texto libre; `content` y `wording` son numéricas continuas.
- `student_id` es único, no hay duplicados de `text`, y todos los `prompt_id` de los resúmenes existen en `prompts_train`.
- No hay valores faltantes en ninguna de las dos tablas.
- Los 4 textos fuente son de temas distintos (tragedia de Aristóteles, Antiguo Egipto, *The Third Wave*, *The Jungle*) y varían en longitud.

## Limpieza

Se normalizan espacios, saltos de línea y caracteres de control. Se conserva la puntuación y las mayúsculas porque son señales de calidad de redacción. Se revisan textos vacíos o muy cortos y los valores faltantes en `content` y `wording`. 

Después se une cada resumen con su texto fuente (`merge` por `prompt_id`) y se calculan las características de texto sobre esa tabla.

In [ ]:
df = summaries_train.copy()
df['text_raw_len'] = df['text'].str.len()
df['text'] = df['text'].map(eu.basic_clean)
df['text_clean_len'] = df['text'].str.len()

print('Textos vacíos tras limpieza:', (df['text'].str.len() == 0).sum())
print('Textos con < 5 palabras   :', (df['text'].str.split().map(len) < 5).sum())
print('Faltantes en objetivos    :\n', df[['content', 'wording']].isna().sum())
df[['text_raw_len', 'text_clean_len']].describe()

In [ ]:
# Revisión de longitudes extremas: los resúmenes más cortos y más largos
wc = df['text'].str.split().map(len)
print('rango de palabras por resumen:', wc.min(), '-', wc.max())
print('\n--- 3 más cortos ---')
for t in df.loc[wc.nsmallest(3).index, 'text']:
    print(repr(t[:300]))
print('\n--- 1 más largo ---')
print(repr(df.loc[wc.idxmax(), 'text'][:400]), '...')

In [ ]:
# Construcción de la tabla de características (resumen + prompt)
feat = eu.build_feature_frame(df, prompts_train)
print(feat.shape)
feat[eu.NUMERIC_FEATURES + eu.TARGETS].head()

## Análisis exploratorio

### Variables numéricas

Estadística descriptiva de las características de texto y de las dos variables objetivo: media, desviación, percentiles, asimetría y curtosis.

In [ ]:
desc = feat[eu.NUMERIC_FEATURES + eu.TARGETS].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc['skew'] = feat[desc.index].skew()
desc['kurtosis'] = feat[desc.index].kurtosis()
desc.round(3)

### Gráficos exploratorios

#### Variables objetivo

In [ ]:
# Boxplots de content y wording (detección de outliers en las variables objetivo)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, col in zip(ax, eu.TARGETS):
    sns.boxplot(y=feat[col], ax=a)
    a.set_title(col)
savefig('box_targets')
plt.show()

# Outliers de las variables objetivo por regla IQR
for col in eu.TARGETS:
    q1, q3 = feat[col].quantile(.25), feat[col].quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((feat[col] < lo) | (feat[col] > hi)).sum()
    print(f'{col}: {n} outliers fuera de [{lo:.2f}, {hi:.2f}]')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, col in zip(ax, eu.TARGETS):
    sns.histplot(feat[col], kde=True, ax=a, bins=40)
    a.set_title(f'Distribución de {col}')
savefig('dist_targets')
plt.show()

In [ ]:
sns.jointplot(data=feat, x='content', y='wording', kind='hex', height=6)
savefig('content_vs_wording')
plt.show()
print('Correlación content-wording:', feat['content'].corr(feat['wording']).round(3))

#### Histogramas de las características

In [ ]:
feat[eu.NUMERIC_FEATURES].hist(figsize=(16, 12), bins=40)
savefig('hist_features')
plt.show()